# Native projection-based embedding
Retain the complete source geometry, particle count, active space, singles ansatz and optimizer budget. The native frame and source0.1 Pauli trimming are explicit; archived energies are historical, and trimmed expectations are not variational bounds for the original Hamiltonian.

Use optional PySCF locally, without a vendor SDK or hosted chemistry service.

In [ ]:
from pathlib import Path

from examples.synthesis.applications.chemistry.projection_based_embedding.projected_based_embedding_tutorial.overview import (
    main,
    model,
    variational,
    workflow,
)

Use the native canonical fermion and qubit representations.

In [ ]:
hartree_to_ev = main.HARTREE_TO_EV

Regenerate the original cc-pVDZ B3LYP water geometry and all24 atomic orbitals.

In [ ]:
molecule = model.regenerate()
print(f"Fresh original DFT energy: {molecule.dft_energy}")

Keep oxygen as fragment, ten fragment electrons, zero environment electrons and three active virtuals.

In [ ]:
print("Fragment / environment electrons: 10 / 0")
print(f"Oxygen weights: {molecule.orbital_weights}")
print(f"Concentric singular values: {molecule.virtual_singular_values}")

Verify the original DFT-in-DFT, trace, zero-leakage and active-space FCI checks.

In [ ]:
assert abs(molecule.density_trace - 10) < 1e-8
assert molecule.active_metric_error < 1e-12
assert molecule.environment_projector_norm == molecule.embedding_potential_norm == 0
print(f"Fresh active FCI energy: {molecule.fci_energy}")

Retain both physical and embedded operator roles, which coincide for this empty environment.

In [ ]:
assert molecule.fermion == molecule.physical_fermion
print(f"Native canonical fermion terms: {len(molecule.fermion.terms)}")

Reuse native Jordan-Wigner mapping, stabilizer reduction and the public optimizer.

In [ ]:
generators = variational.operators(molecule)

Inspect actual freshly generated canonical fermion terms.

In [ ]:
print(molecule.fermion.terms[:5])

Keep all16 source spin-orbital occupation modes.

In [ ]:
print(f"Untapered spin-orbital width: {model.WIDTH}")

Reduce the original two spin-parity symmetries to14 physical bits.

In [ ]:
assert molecule.reduction.n_qubits == 14

Read the Hartree-Fock state in the actual native Clifford frame.

In [ ]:
print(f"Native tapered Hartree-Fock word: {molecule.hf_word}")

Apply the original0.1 Pauli threshold in the declared native frame.

In [ ]:
print(f"Native trimmed terms: {len(molecule.trimmed.terms)}")

Keep the disabled source diagonalization strictly historical.

In [ ]:
historical_disabled_energy = main.ARCHIVED_DISABLED_DIAGONALIZATION
print(f"Historical disabled source reference: {historical_disabled_energy}")

Build all30 spin-preserving singles and verify two nonzero complete states.

In [ ]:
assert len(generators) == 30
state_errors = variational.check_states(molecule, generators)
print(f"Full occupation-state errors: {state_errors}")

Run100 actual bounded native optimizer queries with the original1e-4 tolerance.

In [ ]:
result = workflow.run(molecule)

Read the actual best evaluated parameters without repairing the observed optimizer result.

In [ ]:
print(f"Actual selected parameters: {result.optimization.selected.parameters}")
print(f"Actual best trimmed energy: {result.optimization.selected.energy}")

Keep physical energy evaluation in the same native tapering frame.

In [ ]:
energy_errors = result.energy_errors
assert max(energy_errors) < 1e-9

Apply the same source threshold to the physical Hamiltonian.

In [ ]:
print(f"Native trimmed physical terms: {len(molecule.physical_trimmed.terms)}")

Report actual trimmed and untrimmed physical expectations separately.

In [ ]:
print(f"Trimmed physical estimate: {result.optimization.physical_trimmed_energy}")
print(f"Untrimmed physical energy: {result.optimization.physical_untrimmed_energy}")

Retain the zero environment correction and source WF-in-DFT estimate.

In [ ]:
estimated_total = result.optimization.physical_trimmed_energy + molecule.environment_correction

Compare actual native estimates with fresh full DFT and save the entire query history.

In [ ]:
main.report(result)
main.write(result, Path("projection-embedding-results.json"))